In [ ]:
from collections.abc import Sequence
from random import random, sample, shuffle
from itertools import permutations
from functools import total_ordering

In [ ]:
NUM_CHILDREN = 30
NUM_CHARACTERISTICS = 20

In [ ]:
@total_ordering
class Child:
    _name: str
    _characteristics: tuple[bool]

    def __init__(self, name: str, characteristics: Sequence[bool]):
        self._name = name
        self._characteristics = tuple(characteristics)
    
    def difference(self, other: "Child") -> int:
        if len(self._characteristics) != len(other._characteristics):
            raise ValueError("Children must have the same number of characteristics")
        return sum(s != o for s, o in zip(self._characteristics, other._characteristics))

    def __str__(self) -> str:
        return f"{self._name}:{''.join('+' if c else '-' for c in self._characteristics)}"

    def __repr__(self) -> str:
        return f"{self._name}@{id(self)}"

    def __eq__(self, other):
        return id(self) == id(other)

    def __lt__(self, other):
        return id(self) < id(other)

In [ ]:
children = list()

for c in range(NUM_CHILDREN):
    name = f"Child {c}"
    children.append(Child(name, sample([True, False], k=NUM_CHARACTERISTICS, counts=[NUM_CHARACTERISTICS, NUM_CHARACTERISTICS])))

In [ ]:
for c in children:
    print(c)

In [ ]:
# for solution in permutations(range(1, NUM_CHILDREN)):
    # print([0] + list(solution))

In [ ]:
def fitness(solution: Sequence[Child]) -> int:
    # Total Hamming distance for a cycle anchored at children[0].
    count = 0
    prev_child = children[0]
    for c in solution:
        count += c.difference(prev_child)
        prev_child = c
    count += solution[-1].difference(children[0])
    return count

test = [children[2], children[3], children[4], children[1]]
fit = fitness(test)

print(fit)

for c in test:
    print(c)

In [ ]:
def tweak(solution: Sequence[Child]) -> list[Child]:
    # Minimizes edges (as seen in the 21/10 lesson)
    # ideal when fitness is based on proximity comparisons
    i1, i2 = sorted(sample(range(len(solution)), k=2))
    return list(solution[:i1]) + list(solution[i1:i2 + 1][::-1]) + list(solution[i2 + 1:])

In [ ]:
def normalize(solution: Sequence[Child]) -> list[Child]:
    # A cycle and its reverse have identical fitness.
    # Ex. sol. 0-1-2-3 is the same as 0-3-2-1 as it is in a circle
    if solution[0] > solution[-1]:
        return list(solution[::-1])
    return list(solution)

# Original solution
It is quite inefficient, made before the friday solutions

In [ ]:
MU = 3 # amount of saved solutions
OFFSPRING_COUNT = 3 # amount of generated solutions
GENERATIONS = 10000
MUTATION_RATE = 0.8


def crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    # cycle crossover
    left, right = sorted(sample(range(len(parent1)), k=2))
    segment = list(parent1[left:right + 1])
    remainder = [child for child in parent2 if child not in segment]
    child = remainder[:left] + segment + remainder[left:]
    return normalize(child)


def evolutionary_search() -> list[Child]:
    # 3,3 strategy
    population = [normalize(sample(children[1:], k=len(children) - 1)) for _ in range(MU)]
    best = max(population, key=fitness)

    for _ in range(GENERATIONS):
        offspring = []
        # generate 3 new solutions
        for _ in range(OFFSPRING_COUNT):
            parent1, parent2 = sample(population, k=2)
            candidate = crossover(parent1, parent2)
            if random() < MUTATION_RATE:
                candidate = normalize(tweak(candidate))
            offspring.append(candidate)

        generation_best = max(offspring, key=fitness)
        if fitness(generation_best) > fitness(best):
            best = generation_best
        
        # Since this solution has mu and lambda equal (and the , instead of +)
        # the population is entirely replaced by the offspring
        # changing mu and lambda to not equal values requires modification
        population = offspring

    return best


ea_best = evolutionary_search()

print(f"Evolutionary search: {fitness(ea_best)}")

for child in ea_best:
    print(child)

# New solution

Since technically speaking, the solution I submitted in time is the one above, I do not know if the review has to be only about the one above, but this new solution is a solution meant to address some issues about the old one, mainly these points:
- Recombination always occurs, and mutation is done on top of recombination
- Recombination and mutation always use the same method

Other things might change depending on my choices.

## Support functions

### Mutations

In [ ]:
# More used, as it preserves edges
def inversion_mutation(solution: Sequence[Child]) -> list[Child]:
    i1, i2 = sorted(sample(range(len(solution)), k=2))
    return list(solution[:i1]) + list(solution[i1:i2 + 1][::-1]) + list(solution[i2 + 1:])

In [ ]:
def swap_position(solution: Sequence[Child]) -> list[Child]:
    i1, i2 = sorted(sample(range(len(solution)), k=2))
    sol = solution.copy()
    temp = sol[i1]
    sol[i1] = sol[i2]
    sol[i2] = temp
    return sol

In [ ]:
def scramble_mutation(solution: Sequence[Child]) -> list[Child]:
    sol = solution.copy()

    nInd = 1
    
    # average loci = 4 and always less than half solution length
    while random() < 0.75 and nInd < len(solution)-1:
        nInd += 1

    indexes = []
    chosen = []

    indexes = sample(range(len(solution)), k = nInd)
    for ind in indexes:
        print(ind)
        chosen.append(solution[ind])
        print(chosen)

    # shuffle both index and children, low probability of them being the same, might rework later
    shuffle(chosen)
    shuffle(indexes)

    for index in indexes:
        sol[index] = chosen.pop(0)

    return sol

In [ ]:
def insert_mutation(solution: Sequence[Child]) -> list[Child]:
    i1, i2 = sorted(sample(range(len(solution)), k=2))
    # no change guard
    while abs(i1-i2) < 2:
        i1, i2 = sorted(sample(range(len(solution)), k=2))
    print(i1, i2)
    sol = solution.copy()
    if i2 < i1:
        temp = i1
        i1 = i2
        i2 = temp
    return list(sol[:i1+1]) + list(sol[i2:i2+1]) + list(sol[i1+1:i2]) + list(sol[i2+1:])

In [ ]:
def random_mutation(solution: Sequence[Child]) -> list[Child]:
    chance = random()
    if chance < 0.4:
        return inversion_mutation(solution)
    if chance < 0.6:
        return swap_position(solution)
    if chance < 0.8:
        return scramble_mutation(solution)
    return insert_mutation(solution)

### Crossovers

In [ ]:
def cycle_crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    left, right = sorted(sample(range(len(parent1)), k=2))
    segment = list(parent1[left:right + 1])
    remainder = [child for child in parent2 if child not in segment]
    child = remainder[:left] + segment + remainder[left:]
    return normalize(child)

In [ ]:
def partially_mapped_crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    left, right = sorted(sample(range(len(parent1)), k=2))
    child: list[Child | None] = [None] * len(parent1)
    child[left:right + 1] = parent1[left:right + 1]

    for index in range(left, right + 1):
        gene = parent2[index]
        if gene not in child[left:right + 1]:
            position = index
            while left <= position <= right:
                position = parent2.index(parent1[position])
            child[position] = gene

    for index, gene in enumerate(child):
        if gene is None:
            child[index] = parent2[index]

    return normalize([gene for gene in child if gene is not None])

In [ ]:
# still not right
def inver_over_crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    child = list(parent1)
    current = child[sample(range(len(child)), k=1)[0]]

    

    print(current)
    current_index = 0
    next_index = 0

    while abs(current_index - next_index) < 2:
        parent2_index = parent2.index(current)
        direction = 1 if random() < 0.5 else -1
        print(direction)
        next_gene = parent2[(parent2_index + direction) % len(parent2)]
        current_index = child.index(current)
        next_index = child.index(next_gene)

    left, right = sorted((current_index, next_index))
    if current_index == left:
        child[left+1:right + 1] = child[left+1:right + 1][::-1]
    else:
        child[left:right] = child[left:right][::-1]
    current = next_gene

    return normalize(child)

test = [children[2], children[3], children[6], children[4], children[1], children[5]]
test2 = [children[6], children[4], children[2], children[1], children[5], children[3]]

for c in test:
    print(c)

print("\nParent 2\n")

for c in test2:
    print(c)

print("\nRun\n")

test_cross = inver_over_crossover(test, test2)

print("\nResult\n")

for c in test_cross:
    print(c)

Child 2:--+--+--+-++++++---+
Child 3:+++++-+-++-+-++---+-
Child 6:-+-++---++--+---+++-
Child 4:+--+--+++-+-+-+-++-+
Child 1:-+--+--++---+--+---+
Child 5:--+-+----++---+++-+-

Parent 2

Child 6:-+-++---++--+---+++-
Child 4:+--+--+++-+-+-+-++-+
Child 2:--+--+--+-++++++---+
Child 1:-+--+--++---+--+---+
Child 5:--+-+----++---+++-+-
Child 3:+++++-+-++-+-++---+-

Run

Child 3:+++++-+-++-+-++---+-
-1

Result

Child 6:-+-++---++--+---+++-
Child 4:+--+--+++-+-+-+-++-+
Child 1:-+--+--++---+--+---+
Child 5:--+-+----++---+++-+-
Child 3:+++++-+-++-+-++---+-
Child 2:--+--+--+-++++++---+


## Complete genetic algorithm

The baseline above is kept for comparison. This implementation adds elitism, tournament selection, order-preserving crossover, and inversion mutation. A candidate is the ordering of all children except the fixed first child; `fitness()` closes the circle back to that anchor.

In [ ]:
def order_crossover(parent1: Sequence[Child], parent2: Sequence[Child]) -> list[Child]:
    """Order crossover for two permutations of the same children."""
    if len(parent1) != len(parent2) or set(parent1) != set(parent2):
        raise ValueError("Parents must contain the same children")
    n = len(parent1)
    if n < 2:
        return list(parent1)

    left, right = sorted(sample(range(n), k=2))
    child: list[Child | None] = [None] * n
    child[left:right + 1] = parent1[left:right + 1]
    remaining = [gene for gene in parent2 if gene not in child]
    positions = list(range(right + 1, n)) + list(range(0, left))
    for position, gene in zip(positions, remaining):
        child[position] = gene
    return [gene for gene in child if gene is not None]


def inversion_mutation_safe(solution: Sequence[Child]) -> list[Child]:
    if len(solution) < 2:
        return list(solution)
    left, right = sorted(sample(range(len(solution)), k=2))
    return list(solution[:left]) + list(solution[left:right + 1][::-1]) + list(solution[right + 1:])


def improved_evolutionary_search(
    population_size: int = 100,
    generations: int = 1000,
    mutation_rate: float = 0.2,
    elite_count: int = 2,
):
    """Maximize total circular Hamming distance; keep the best candidate each generation."""
    if not children:
        raise ValueError("At least one child is required")
    if population_size < 2:
        raise ValueError("population_size must be at least 2")
    if generations < 0:
        raise ValueError("generations cannot be negative")
    if not 0 <= mutation_rate <= 1:
        raise ValueError("mutation_rate must be between 0 and 1")
    if not 1 <= elite_count < population_size:
        raise ValueError("elite_count must be between 1 and population_size - 1")

    genes = children[1:]  # children[0] anchors the circular arrangement
    if len(genes) < 2:
        return list(genes)

    population = [sample(genes, k=len(genes)) for _ in range(population_size)]
    best = max(population, key=fitness)

    for _ in range(generations):
        ranked = sorted(population, key=fitness, reverse=True)
        next_population = [list(candidate) for candidate in ranked[:elite_count]]

        while len(next_population) < population_size:
            # Small tournaments favor fitter candidates while retaining diversity.
            contestants = sample(population, k=min(3, len(population)))
            parent1 = max(contestants, key=fitness)
            contestants = sample(population, k=min(3, len(population)))
            parent2 = max(contestants, key=fitness)
            candidate = order_crossover(parent1, parent2)
            if random() < mutation_rate:
                candidate = inversion_mutation_safe(candidate)
            next_population.append(candidate)

        population = next_population
        generation_best = max(population, key=fitness)
        if fitness(generation_best) > fitness(best):
            best = list(generation_best)

    return list(best)


# Run this cell to use the improved search.
improved_best = improved_evolutionary_search()
print(f"Best circular Hamming distance: {fitness(improved_best)}")
for child in improved_best:
    print(child)
